Section 2.4 Content-based recs

In [24]:
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity

from recsys.data.loaders import load_movielens
from recsys.utils.colab import get_data_path


In [2]:

DATA_PATH = get_data_path()
ratings, movies = load_movielens("ml-25m", data_dir=DATA_PATH)
ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [3]:
# Keep ratings/movies loaded from load_movielens in Cell 2
print(ratings.head())
print(f"ratings shape: {ratings.shape}")
print(movies.head())
print(f"movies shape: {movies.shape}")

   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
1       1      306     3.5  1147868817
2       1      307     5.0  1147868828
3       1      665     5.0  1147878820
4       1      899     3.5  1147868510
ratings shape: (25000095, 4)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
movies shape: (62423, 3)


In [4]:
from scipy.sparse import csr_matrix

user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()
user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)} #A
movie_to_idx = {mid: idx for idx, mid in enumerate(movie_ids)} #A
idx_to_movie = {idx: mid for mid, idx in movie_to_idx.items()} #A

rows = [user_to_idx[uid] for uid in ratings['userId']] #B
cols = [movie_to_idx[mid] for mid in ratings['movieId']] #B
data = [1] * len(ratings) #B

user_item_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(user_ids), len(movie_ids))
) #C
print(f"user_item_matrix shape: {user_item_matrix.shape}")

user_item_matrix shape: (162541, 59047)


In [5]:
movie_counts = ratings['movieId'].value_counts() #A
max_count = movie_counts.max() #B

popularity_scores = {
    int(mid): float(count / max_count)
    for mid, count in movie_counts.items()
}


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(user_item_matrix.T, dense_output=False) #A
print(f"Similarity matrix shape: {item_similarity.shape}")
print(f"Similarity matrix memory: {item_similarity.data.nbytes / 1024 / 1024:.1f} MB")


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
movies["clean_title"] = (
    movies["title"]
    .str.replace(r"\s*\((?:19|20)\d{2}(?:-(?:19|20)\d{2})?\)\s*$", "", regex=True)
    .str.strip()
)
movies['content'] = movies['clean_title'] + ' ' + movies['genres'].fillna('')

vectorizer = TfidfVectorizer(
    max_features=500,
    stop_words='english',
    token_pattern=r'(?u)\b\w+\b'
)

content_vectors = vectorizer.fit_transform(movies['content'])

print(f"Content vectors shape: {content_vectors.shape}")
print(f"Memory: {content_vectors.data.nbytes / 1024 / 1024:.1f} MB")

print(f"\nTop features (most distinctive words):")
feature_names = vectorizer.get_feature_names_out()

for word in feature_names[:10]:
    print(f"  {word}")

idf_scores = vectorizer.idf_
# Top 10 most distinctive words
top_idx = np.argsort(idf_scores)[-10:][::-1]

print("Top 10 most distinctive words (highest IDF):")
for word, score in zip(feature_names[top_idx], idf_scores[top_idx]):
    print(f"{word}: {score:.4f}")

Content vectors shape: (62423, 500)
Memory: 1.3 MB

Top features (most distinctive words):
  1
  10
  100
  11
  12
  13
  2
  3
  4
  5
Top 10 most distinctive words (highest IDF):
bang: 8.5760
f: 8.3040
fast: 8.2805
zombies: 8.2805
youth: 8.2805
louis: 8.2805
passion: 8.2805
anna: 8.2805
12: 8.2805
touch: 8.2575


In [15]:
movies['content']


0        Toy Story Adventure|Animation|Children|Comedy|...
1                       Jumanji Adventure|Children|Fantasy
2                          Grumpier Old Men Comedy|Romance
3                   Waiting to Exhale Comedy|Drama|Romance
4                       Father of the Bride Part II Comedy
                               ...                        
62418                                             We Drama
62419                       Window of the Soul Documentary
62420                               Bad Poems Comedy|Drama
62421                      A Girl Thing (no genres listed)
62422       Women of Devil's Island Action|Adventure|Drama
Name: content, Length: 62423, dtype: object

In [20]:
# If you want rows where the content text contains the word "genres"
rows_with_genres_word = movies[movies["content"].str.contains(r"\bgenres\b", case=False, na=False)]
rows_with_genres_word.head()

,movieId,title,genres,content,clean_title
15881,83773,Away with Words (San tiao ren) (1999),(no genres listed),Away with Words (San tiao ren) (no genres listed),Away with Words (San tiao ren)
16060,84768,Glitterbug (1994),(no genres listed),Glitterbug (no genres listed),Glitterbug
16351,86493,"Age of the Earth, The (A Idade da Terra) (1980)",(no genres listed),"Age of the Earth, The (A Idade da Terra) (no g...","Age of the Earth, The (A Idade da Terra)"
16491,87061,Trails (Veredas) (1978),(no genres listed),Trails (Veredas) (no genres listed),Trails (Veredas)
17404,91246,Milky Way (Tejút) (2007),(no genres listed),Milky Way (Tejút) (no genres listed),Milky Way (Tejút)


In [22]:
movie_id_to_idx = {
    int(mid): idx 
    for idx, mid in enumerate(movies['movieId'])
} #A
idx_to_movie_id = {idx: mid for mid, idx in movie_id_to_idx.items()} #A

def retrieve_similar_by_content(movie_id, k=100):
    if movie_id not in movie_id_to_idx:
        return []
    
    movie_idx = movie_id_to_idx[movie_id] #B
    movie_vector = content_vectors[movie_idx] #B
    
    similarities = cosine_similarity(movie_vector, content_vectors)[0] #C
    
    top_indices = np.argsort(similarities)[-(k+1):-1][::-1] #D
    
    candidates = []
    for idx in top_indices:
        candidates.append({
            'movie_id': int(idx_to_movie_id[idx]),
            'content_similarity': float(similarities[idx])
        })
    
    return candidates


In [52]:
input_movie_id = 209157
candidates = retrieve_similar_by_content(input_movie_id, k=10)
title = movies.loc[movies['movieId'] == input_movie_id, 'title'].values[0]
print(f"Input movie, ID: {input_movie_id}, Title: {title}")
print("Similar movies based on content:")
for i, item in enumerate(candidates, 1):
    title = movies[movies['movieId'] == item['movie_id']]['title'].values[0]
    score = item['content_similarity']
    genres = movies[movies['movieId'] == item['movie_id']]['genres'].values[0]
    
    print(f"{item['movie_id']}.\t {title}, sim: {score:.4f}, genres: {genres}")


    

Input movie, ID: 209157, Title: We (2018)
Similar movies based on content:
160149.	 Complete Unknown (2016), sim: 1.0000, genres: Drama
5158.	 Cross Creek (1983), sim: 1.0000, genres: Drama
137471.	 Tracks (1977), sim: 1.0000, genres: Drama
188619.	 Denmark (2017), sim: 1.0000, genres: Drama
171157.	 Lowriders (2017), sim: 1.0000, genres: Drama
188605.	 The Sower (2017), sim: 1.0000, genres: Drama
199656.	 Río Escondido (1948), sim: 1.0000, genres: Drama
137494.	 Who Killed Pasolini? (1995), sim: 1.0000, genres: Drama
137561.	 Saturn in Opposition (2007), sim: 1.0000, genres: Drama
5237.	 Taps (1981), sim: 1.0000, genres: Drama


In [41]:
movies[movies['movieId'] == 209157]


,movieId,title,genres,content,clean_title
62418,209157,We (2018),Drama,We Drama,We


In [62]:
def get_user_history(user_id, k=None) -> int: #A
    user_rows = ratings[ratings["userId"] == user_id]
    user_rows = user_rows.sort_values("timestamp")
    movie_ids = user_rows["movieId"].tolist()
    return set(movie_ids) if k is None else set(movie_ids[-k:])


def filter_watched(candidates, user_id) -> list[dict]: #B
    user_history = get_user_history(user_id)
    
    filtered = [
        item for item in candidates
        if item['movie_id'] not in user_history
    ]    
    return filtered
def score_popularity(candidates):
    for item in candidates:
        item['popularity'] = popularity_scores.get(item['movie_id'], 0.0)
    return candidates


def recommend_content_based(user_id, k=10, content_weight=0.7):
  user_history = get_user_history(user_id)
    
  if len(user_history) == 0:
    popular_movies = movie_counts.head(k).index.tolist()
    return [{'movie_id': int(mid)} for mid in popular_movies]
    
  all_candidates = {}
  recent_movies = list(user_history)[-20:]
    
  for movie_id in recent_movies:
    candidates = retrieve_similar_by_content(movie_id, k=50)
        
    for item in candidates:
      mid = item['movie_id']
      score = item['content_similarity']
      if mid in all_candidates:
        all_candidates[mid] = max(all_candidates[mid], score)
      else:
        all_candidates[mid] = score
    
  candidates = [
    {'movie_id': mid, 'content_similarity': score}
    for mid, score in all_candidates.items()
  ]
  candidates = filter_watched(candidates, user_id)

  candidates = score_popularity(candidates)
      
  for item in candidates:
    item['final_score'] = (
      content_weight * item['content_similarity'] +
      (1 - content_weight) * item['popularity']
      )
    
  ranked = sorted(candidates, key=lambda x: x['final_score'], reverse=True)
    
  return ranked[:k]


In [64]:
recs = recommend_content_based(user_id=1, k=10, content_weight=0.7)
for i, item in enumerate(recs, 1):
    title = movies[movies['movieId'] == item['movie_id']]['title'].values[0]
    score = item['content_similarity']
    genres = movies[movies['movieId'] == item['movie_id']]['genres'].values[0]
    
    print(f"{item['movie_id']}.\t {title}, sim: {score:.4f}, genres: {genres}")


    

1.	 Toy Story (1995), sim: 0.8862, genres: Adventure|Animation|Children|Comedy|Fantasy
1270.	 Back to the Future (1985), sim: 0.8552, genres: Adventure|Comedy|Sci-Fi
7254.	 The Butterfly Effect (2004), sim: 1.0000, genres: Drama|Sci-Fi|Thriller
8961.	 Incredibles, The (2004), sim: 0.9165, genres: Action|Adventure|Animation|Children|Comedy
115713.	 Ex Machina (2015), sim: 1.0000, genres: Drama|Sci-Fi|Thriller
1921.	 Pi (1998), sim: 1.0000, genres: Drama|Sci-Fi|Thriller
68954.	 Up (2009), sim: 0.9125, genres: Adventure|Animation|Children|Drama
33615.	 Madagascar (2005), sim: 1.0000, genres: Adventure|Animation|Children|Comedy
2431.	 Patch Adams (1998), sim: 1.0000, genres: Comedy|Drama
6979.	 WarGames (1983), sim: 1.0000, genres: Drama|Sci-Fi|Thriller


In [ ]:
import importlib

import recsys
importlib.reload(recsys.fourstage_recsys.retrieval.itemknn_retrieval)
importlib.reload(recsys.fourstage_recsys.filtering.history_filtering)
from recsys.fourstage_recsys.retrieval.itemknn_retrieval import ItemKNNRetrieval
from recsys.fourstage_recsys.filtering.history_filtering import HistoryFiltering

item_knn_retrieval = ItemKNNRetrieval(ratings)
history_filtering = HistoryFiltering(ratings)

def retrieve_hybrid(movie_id, k=100):
    content_candidates = retrieve_similar_by_content(movie_id, k=k//2)
    behavioral_candidates = item_knn_retrieval.retrieve_similar_items(movie_id, k=k//2)
    
    all_candidates = {}
    
    for item in content_candidates:
        mid = item['movie_id']
        all_candidates[mid] = {
            'movie_id': mid,
            'content_similarity': item['content_similarity'],
            'behavioral_similarity': 0.0
        }
    
    for item in behavioral_candidates:
        mid = item['movie_id']
        if mid in all_candidates:
            all_candidates[mid]['behavioral_similarity'] = item['similarity']
        else:
            all_candidates[mid] = {
                'movie_id': mid,
                'content_similarity': 0.0,
                'behavioral_similarity': item['similarity']
            }
    
    return list(all_candidates.values())

retrieve_hybrid(1, k=10)

AttributeError: module 'recsys.fourstage_recsys' has no attribute 'filtering'

In [ ]:

def rank_three_signals(candidates, content_weight=0.3, behavioral_weight=0.5, k=10):
    popularity_weight = 1.0 - content_weight - behavioral_weight
    
    for item in candidates:
        item['final_score'] = (
            content_weight * item.get('content_similarity', 0) +
            behavioral_weight * item.get('behavioral_similarity', 0) +
            popularity_weight * item.get('popularity', 0)
        )
    
    ranked = sorted(candidates, key=lambda x: x['final_score'], reverse=True)
    return ranked[:k]

seed_movie_id = 1
candidates = retrieve_hybrid(seed_movie_id, k=100)
candidates = add_popularity_scores(candidates)

ranked = rank_three_signals(
    candidates,
    content_weight=0.3,
    behavioral_weight=0.5
)

print("Hybrid recommendations (content + behavior + popularity):\n")
for i, item in enumerate(ranked[:5], 1):
    title = movies[movies['movieId'] == item['movie_id']]['title'].values[0]
    print(f"{i}. {title}")
    print(f"   Content: {item.get('content_similarity', 0):.3f}")
    print(f"   Behavior: {item.get('behavioral_similarity', 0):.3f}")
    print(f"   Popularity: {item.get('popularity', 0):.3f}")
    print(f"   Final: {item['final_score']:.3f}\n")


NameError: name 'retrieve_similar_items' is not defined

In [ ]:
from recsys import FourStageRecommender #A
from recsys.retrievals import ItemKNNRetrieval #B
from recsys.filters import HistoryFilter #B
from recsys.scorers import PopularityScorer #B
from recsys.rankers import WeightedRanker #B

retrieval = ItemKNNRetrieval(
    interactions_path='data/ratings.csv',
    items_path='data/movies.csv'
) #C

recommender = FourStageRecommender(
  retrieval=retrieval,
  filter=HistoryFilter(),          
  scorer=PopularityScorer(interactions_path='data/ratings.csv'),
  ranker=WeightedRanker(weights={'similarity': 0.7, 'popularity': 0.3})
) #D

recommendations = recommender.recommend(
    user_id='123',
    k=10
) #H
